# Imports

In [1]:
import torch
import torchvision
from torchvision.models import resnet18, densenet121
from torchvision.transforms import v2
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import random

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cuda


In [3]:
from google.colab import drive
import os

drive.mount('/content/drive')

default_notebook_path = '/content/drive/MyDrive/Kristiania/5th_Semester/Deep_Learning/Mandatory_Assignment'
if os.path.exists(default_notebook_path):
    cifar10data = os.path.join(default_notebook_path, 'cifar10_data')
else:
    cifar10data = '/content/drive/MyDrive/Kristiania/5th_Semester/Deep_Learning/Mandatory_Assignment/cifar10_data'

os.makedirs(cifar10data, exist_ok=True)
print(f"CIFAR-10 datasets will be saved/loaded from: {cifar10data}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CIFAR-10 datasets will be saved/loaded from: /content/drive/MyDrive/Kristiania/5th_Semester/Deep_Learning/Mandatory_Assignment/cifar10_data


# Defining Three Architectures

In [4]:
# (I'll remove name later)
# Note for julien: Baseline, no change needed
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

# Note for julien: I'll change this one
def get_cifar_resnet18():
    model = resnet18(weights=None, num_classes=10)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)            # Replace the initial 7x7 convolution with a smaller 3x3 convolution
    model.maxpool = nn.Identity()                                                             # Remove the max pooling layer
    return model

# Note for julien: You change this one
def get_cifar_densenet121():
    model = densenet121(weights=None, num_classes=10)
    model.features.conv0 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)   # Modify the first convolutional layer and pooling for 32x32 inputs
    model.features.pool0 = nn.Identity()
    return model

architectures = {
    "baseline_simple_cnn": SimpleCNN,
    "resnet18": get_cifar_resnet18,
    "densenet121": get_cifar_densenet121
}

# Defining Three Augmentations

In [5]:
base_ops = [
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
]

augmentation_strategies = {
    # Note for julien: Baseline, no change needed
    "baseline_augmentation": v2.Compose(base_ops),

    # Note for julien: I'll change this one
    "aug1": v2.Compose([
        v2.ToImage(),
        v2.RandomCrop(32, padding=4),
        v2.RandomHorizontalFlip(),
        *base_ops
    ]),

    # Note for julien: You change this one
    "aug2": v2.Compose([
        v2.ToImage(),
        v2.RandomCrop(32, padding=4),
        v2.RandomHorizontalFlip(),
        v2.RandAugment(num_ops=2, magnitude=9),
        *base_ops
    ])
}

# Defining Three Optimizers

In [6]:
optimizers = {
    # Note for julien: Baseline, no change needed
    "baseline_sgd_momentum": lambda params: optim.SGD(params, lr=0.01, momentum=0.9),
    # Note for julien: I'll change this one
    "adam": lambda params: optim.Adam(params, lr=0.01),
    # Note for julien: You change this one
    "adamw": lambda params: optim.AdamW(params, lr=0.01, weight_decay=0.01)
}

# Selecting Configuration

In [7]:
train_transform = augmentation_strategies["aug1"]
current_architecture = architectures["resnet18"]
current_optimizer = optimizers["adamw"]

# 1 Pre Processing

## 1.2 Seeding

In [8]:
experiment_seeds = [0, 1, 2]

def set_seed(seed):
    torch.manual_seed(seed)             # CPU Seed for Torch
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)    # GPU Seed for Torch

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

current_seed = experiment_seeds[0]
set_seed(current_seed)

## 1.3 CIFAR-10 Dataset

### 1.3.1 Downloading and Tranforming

In [9]:
# Assign the transforms to the training set
train_transform = train_transform


# Transform for validation and test data (NO augmentations)
eval_transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

full_trainset_aug = torchvision.datasets.CIFAR10(root=cifar10data, train=True, download=True, transform=train_transform)
full_trainset_eval = torchvision.datasets.CIFAR10(root=cifar10data, train=True, download=True, transform=eval_transform)

split_generator = torch.Generator().manual_seed(42)                 # Hard coding Seeding for generator (Should be the same every time)
indices = torch.randperm(50000, generator=split_generator).tolist()
train_idx, val_idx = indices[:45000], indices[45000:]

trainset = torch.utils.data.Subset(full_trainset_aug, train_idx)    # map the training indices to the augmented data
valset = torch.utils.data.Subset(full_trainset_eval, val_idx)       # map the valuation indices to the normalized data

# The test set only uses the evaluation transform
testset = torchvision.datasets.CIFAR10(root=cifar10data, train=False, download=True, transform=eval_transform)

In [10]:
batch_size = 64
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)
valloader = torch.utils.data.DataLoader(valset, batch_size=batch_size, shuffle=False, num_workers=2)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)

### 1.3.2 Step by Step Inspection for understanding underlying data

In [11]:
pretransform = torchvision.datasets.CIFAR10(root=cifar10data, train=True, download=True, transform=None)

In [12]:
print("--Before Transformation Shape and Dtype--\n ---------------------------------------")
pimg, plabel = pretransform[0]
print(type(pimg))
print(f"Size (W, H): {pimg.size}")
print(f"Mode: {pimg.mode}")

print("\n--After Transformation Shape and Dtype--\n ---------------------------------------")
img, label = trainset[0]
print(type(img))
print(img.shape)
print(img.dtype)

step1_to_image = v2.ToImage()
step2_to_dtype = v2.ToDtype(torch.float32, scale=True)
step3_normalize = v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))

img_t1 = step1_to_image(pimg)
img_t2 = step2_to_dtype(img_t1)
img_t3 = step3_normalize(img_t2)

print("\n-- Step-by-Step --")
print(f"After ToImage:   type={type(img_t1)}, shape={img_t1.shape}, dtype={img_t1.dtype}, min={img_t1.min()}, max={img_t1.max()}")
print(f"After ToDtype:   type={type(img_t2)}, shape={img_t2.shape}, dtype={img_t2.dtype}, min={img_t2.min():.2f}, max={img_t2.max():.2f}")
print(f"After Normalize: type={type(img_t3)}, shape={img_t3.shape}, dtype={img_t3.dtype}, min={img_t3.min():.2f}, max={img_t3.max():.2f}")

--Before Transformation Shape and Dtype--
 ---------------------------------------
<class 'PIL.Image.Image'>
Size (W, H): (32, 32)
Mode: RGB

--After Transformation Shape and Dtype--
 ---------------------------------------
<class 'torchvision.tv_tensors._image.Image'>
torch.Size([3, 32, 32])
torch.float32

-- Step-by-Step --
After ToImage:   type=<class 'torchvision.tv_tensors._image.Image'>, shape=torch.Size([3, 32, 32]), dtype=torch.uint8, min=0, max=255
After ToDtype:   type=<class 'torchvision.tv_tensors._image.Image'>, shape=torch.Size([3, 32, 32]), dtype=torch.float32, min=0.00, max=1.00
After Normalize: type=<class 'torchvision.tv_tensors._image.Image'>, shape=torch.Size([3, 32, 32]), dtype=torch.float32, min=-1.00, max=1.00


In [13]:
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Hyperparameters & Configuration
epochs = 30
patience = 6              # Early stopping patience
scheduler_patience = 3    # LR scheduler patience
lr_factor = 0.1           # Multiply LR by this when val loss plateaus

# Vi oppretter tomme lister for å lagre historikken
train_losses = []
val_losses = []
train_accs = []
val_accs = []

# Nullstiller modell og optimalisator for en ny ren kjøring
model = current_architecture().to(device)
optimiser = current_optimizer(model.parameters())
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimiser, mode='min', factor=lr_factor, patience=scheduler_patience)

best_val_loss = float('inf')
patience_counter = 0

print("Starter trening med loggføring av historikk...")
for epoch in range(epochs):
    # Training Phase
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for inputs, labels in trainloader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimiser.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimiser.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total_train += labels.size(0)
        correct_train += predicted.eq(labels).sum().item()

    average_train_loss = running_loss / len(trainloader)
    train_accuracy = 100.0 * correct_train / total_train

    # Validation Phase
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for inputs, labels in valloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            _, predicted = outputs.max(1)
            total_val += labels.size(0)
            correct_val += predicted.eq(labels).sum().item()

    average_val_loss = val_loss / len(valloader)
    val_accuracy = 100.0 * correct_val / total_val

    # Save for plotting purposes
    train_losses.append(average_train_loss)
    val_losses.append(average_val_loss)
    train_accs.append(train_accuracy)
    val_accs.append(val_accuracy)

    scheduler.step(average_val_loss)
    current_lr = optimiser.param_groups[0]['lr']

    print(f"Epoch {epoch+1:02d}/{epochs:02d} | "
          f"Train Loss: {average_train_loss:.4f} | "
          f"Val Loss: {average_val_loss:.4f} | "
          f"Val Acc: {val_accuracy:.2f}% | "
          f"LR: {current_lr}")

    # Early Stop Mechanic
    if average_val_loss < best_val_loss:
        best_val_loss = average_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), 'best_model.pth')
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\nEarly stopping aktivert etter {patience} epoker uten forbedring.")
            model.load_state_dict(torch.load('best_model.pth'))
            break

Starter trening med loggføring av historikk...
Epoch 01/30 | Train Loss: 1.7784 | Val Loss: 1.6089 | Val Acc: 41.78% | LR: 0.01
Epoch 02/30 | Train Loss: 1.2911 | Val Loss: 1.1546 | Val Acc: 58.76% | LR: 0.01
Epoch 03/30 | Train Loss: 1.0075 | Val Loss: 1.0964 | Val Acc: 61.82% | LR: 0.01
Epoch 04/30 | Train Loss: 0.8385 | Val Loss: 0.8967 | Val Acc: 69.54% | LR: 0.01
Epoch 05/30 | Train Loss: 0.7054 | Val Loss: 0.7322 | Val Acc: 75.60% | LR: 0.01
Epoch 06/30 | Train Loss: 0.6106 | Val Loss: 0.6154 | Val Acc: 78.50% | LR: 0.01
Epoch 07/30 | Train Loss: 0.5396 | Val Loss: 0.6333 | Val Acc: 79.70% | LR: 0.01
Epoch 08/30 | Train Loss: 0.4964 | Val Loss: 0.5256 | Val Acc: 81.92% | LR: 0.01
Epoch 09/30 | Train Loss: 0.4539 | Val Loss: 0.4900 | Val Acc: 83.02% | LR: 0.01
Epoch 10/30 | Train Loss: 0.4236 | Val Loss: 0.4554 | Val Acc: 84.28% | LR: 0.01
Epoch 11/30 | Train Loss: 0.3934 | Val Loss: 0.4694 | Val Acc: 84.06% | LR: 0.01
Epoch 12/30 | Train Loss: 0.3693 | Val Loss: 0.4173 | Val Acc:

In [ ]:
'''import numpy as np
import copy

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparameters
epochs = 30
patience = 6
scheduler_patience = 3
lr_factor = 0.1

# Define the OFAT configurations
configurations = {
    "Baseline": {"arch": "baseline_simple_cnn", "aug": "baseline_augmentation", "opt": "baseline_sgd_momentum"},
    "ResNet18": {"arch": "resnet18", "aug": "baseline_augmentation", "opt": "baseline_sgd_momentum"},
    "DenseNet121": {"arch": "densenet121", "aug": "baseline_augmentation", "opt": "baseline_sgd_momentum"},
    "Aug1": {"arch": "baseline_simple_cnn", "aug": "aug1", "opt": "baseline_sgd_momentum"},
    "Aug2": {"arch": "baseline_simple_cnn", "aug": "aug2", "opt": "baseline_sgd_momentum"},
    "Adam": {"arch": "baseline_simple_cnn", "aug": "baseline_augmentation", "opt": "adam"},
    "AdamW": {"arch": "baseline_simple_cnn", "aug": "baseline_augmentation", "opt": "adamw"}
}

results = {}

for config_name, config in configurations.items():
    print(f"\n{'='*50}\nTesting Configuration: {config_name}\n{'='*50}")

    # Update transform dynamically for this configuration
    full_trainset_aug.transform = augmentation_strategies[config["aug"]]

    test_accuracies = []

    for seed in experiment_seeds:
        print(f"\n--- Training Seed {seed} ---")
        set_seed(seed)

        # Re-initialize DataLoaders to reset workers and RNG
        trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)
        valloader = torch.utils.data.DataLoader(valset, batch_size=batch_size, shuffle=False, num_workers=2)
        testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)

        # Initialize Architecture & Optimizer
        model = architectures[config["arch"]]().to(device)
        optimiser = optimizers[config["opt"]](model.parameters())
        criterion = nn.CrossEntropyLoss()
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimiser, mode='min', factor=lr_factor, patience=scheduler_patience)

        best_val_loss = float('inf')
        patience_counter = 0
        best_model_state = None

        for epoch in range(epochs):
            # Training Phase
            model.train()
            running_loss = 0.0
            for inputs, labels in trainloader:
                inputs, labels = inputs.to(device), labels.to(device)
                optimiser.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimiser.step()
                running_loss += loss.item()

            # Validation Phase
            model.eval()
            val_loss = 0.0
            correct_val = 0
            total_val = 0
            with torch.no_grad():
                for inputs, labels in valloader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    val_loss += loss.item()
                    _, predicted = outputs.max(1)
                    total_val += labels.size(0)
                    correct_val += predicted.eq(labels).sum().item()

            avg_val_loss = val_loss / len(valloader)
            scheduler.step(avg_val_loss)

            # Early Stopping
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                patience_counter = 0
                best_model_state = copy.deepcopy(model.state_dict())
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f"Early stopping triggered at epoch {epoch+1}")
                    break

        # Testing Phase (on the best weights)
        model.load_state_dict(best_model_state)
        model.eval()
        correct_test = 0
        total_test = 0
        with torch.no_grad():
            for inputs, labels in testloader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, predicted = outputs.max(1)
                total_test += labels.size(0)
                correct_test += predicted.eq(labels).sum().item()

        test_acc = 100.0 * correct_test / total_test
        test_accuracies.append(test_acc)
        print(f"Final Test Accuracy for {config_name} (Seed {seed}): {test_acc:.2f}%")

    results[config_name] = test_accuracies

print("\n=== FINAL RESULTS ===")
for config_name, accs in results.items():
    print(f"{config_name}: Mean = {np.mean(accs):.2f}%, Std = {np.std(accs):.2f}%")'''